# GEM-XAI final run (v2): news sentiment + revised validation + ablations

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then **Runtime → Run all**.

When asked, upload **`gem-xai-v2.zip`**. Total time is roughly 1 hour. Keep this tab open.

At the end, **`results_v2.zip`** downloads automatically. Send that file back.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch on the T4 GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/gemxai_v2
%cd /content/drive/MyDrive/gemxai_v2

In [ ]:
import os
from google.colab import files
if not os.path.exists('gem-xai/run_all.py'):
    up = files.upload()   # choose gem-xai-v2.zip
    !unzip -o -q gem-xai-v2.zip
%cd gem-xai
!ls

In [ ]:
!pip install -q hmmlearn shap captum grad-cam optuna yfinance mplfinance accelerate

## 1. Main experiment (about 40–50 minutes)
* real Guardian/FinBERT sentiment (`cache/daily_sentiment.csv`)
* models trained on 2010–2021, validated on 2022–2023 (all three regimes present), tested on Jan 2024 – Jun 2025 (unchanged)
* 40 Optuna trials (learning rate, dropout, modality dropout), 5 seeds, modality ablations, bootstrap CIs, calibration, XGBoost SHAP

In [ ]:
!python run_all.py --news-csv cache/daily_sentiment.csv --model-train-end 2021-12-31 --val-end 2023-12-31 --optuna-trials 40 --seeds 5 --out results_v2 2>&1 | tee results_v2_log.txt

## 2. LLM explanations with an open model (about 15–25 minutes)
Uses Qwen2.5-3B-Instruct (free, no account needed) on the same 50 evidence blocks, with the faithfulness verifier.

In [ ]:
!python run_llm_only.py --results results_v2 --llm-backend hf --hf-model Qwen/Qwen2.5-3B-Instruct 2>&1 | tee -a results_v2_log.txt

## 3. Download the results

In [ ]:
!cp results_v2_log.txt results_v2/
!zip -q -r results_v2.zip results_v2
from google.colab import files
files.download('results_v2.zip')
print(open('results_v2/tables/table_test_results.csv').read())